# PARKED — do not run this notebook expecting it to work

**Status: parked, unvalidated, incomplete.** Fine-tuning was stopped before this notebook
was finished or executed once. It is checked in as-is so the work is not lost, not because
it is ready.

**Why it is parked.** The owner wants **vanilla (no fine-tune) numbers first** — what an
off-the-shelf small model scores on the frame task under a grammar constraint, with no
LoRA anywhere. Until that baseline exists there is nothing to compare a fine-tune against,
and a fine-tuned number on its own cannot tell you whether the training helped or whether
the grammar constraint did all the work.

**What is true about it.** The shared, model-independent half was written carefully and
several pieces were checked locally against the repository at commit `11c9c796`:

* the frame JSON Schema accepts **434/434** gold wire frames and 8 000/8 000 sampled
  training frames (also extracted to `frame_schema.py`);
* the corpus verification numbers in the markdown below were **measured**, not estimated
  (also extracted to `verify_corpus.py`);
* the model lineup in `MODELS.md` was verified against the Hugging Face API, not recalled.

**What was never done.** This notebook has **never been executed**. Nothing below has run
on a GPU. The sweep driver, the comparison table and the optional results-push cells were
not written at all — the notebook ends after the per-arm helper definitions. Treat every
cell as a draft.

**What to reuse when fine-tuning restarts** (all beside this file, standalone, with no
notebook dependencies):

| file | what it is |
| --- | --- |
| [`frame_schema.py`](frame_schema.py) | flat frame -> JSON Schema, derived from `frame.py` + `lexicon.py`, with its own proof |
| [`lora_targets.py`](lora_targets.py) | per-architecture LoRA target-module resolution by module inspection, with the trainable-fraction assertion |
| [`verify_corpus.py`](verify_corpus.py) | the training-corpus verification harness and its drop-rate report |
| [`MODELS.md`](MODELS.md) | the 2026-09 small on-device model landscape, with licences and sources |
| [`KAGGLE.md`](KAGGLE.md) | how the notebook was meant to be run, and every spot expected to need a fix |


# Qwen on mobile — fine-tune a small Qwen into Centraid's **frame** task

The model never writes SQL and never writes a canonical character. It reads one sentence a
person said to their own private vault and fills a flat **frame** — `head / set / filters /
window / orderBy / limit / ref / combineOp / cmds / refuseReason`. Repository code
(`experiments/afm-spike/frame.py`) renders that frame to the canonical grammar, and the
repository's own executor (`run-model`) scores it. Nothing in this notebook repairs a model
output; a malformed, empty or unrenderable frame is a **failed turn**, counted as one.

**Target: mobile.** The deliverable is a Q4_K_M GGUF small enough to ship on a phone, plus
the accuracy-vs-epoch curve that says what that size buys.

**The floor to beat** — Tier D, the hand-written rule parser with no model at all:

| corpus | strict sessions |
| --- | --- |
| suite | **53 / 120** |
| blind | **34 / 60** |
| holdout | **31 / 78** |

Two earlier model families failed the same way: trained on the TEMPLATE corpus
(`data/train.jsonl`) they learned the template *generator*, not English — flan-t5-base
scored 41.4% on held-out templates and **0.0%** on real utterances. This notebook trains on
`data/distill.jsonl` (45 779 LLM-written paraphrases over a second invented world, held out
by *skeleton*), not on the template corpus.

Read `experiments/qwen-mobile/KAGGLE.md` before running. Everything tunable is in the next
cell; nothing below it needs editing.


In [ ]:
# ============================== CONFIG ==============================
# The ONLY cell you edit. Everything below reads from here.

# ---- where the code comes from -------------------------------------
GIT_HOST        = "github.com/srikanth235/centraid"
GIT_BRANCH      = "claude/friendly-cori-ezadjb-wip-canon"
GIT_COMMIT      = "11c9c7969074c6c91f6ba941cff1361af8c44fd1"   # asserted after clone
ALLOW_COMMIT_DRIFT = False        # True = warn instead of abort if the branch moved
KAGGLE_SECRET   = "CENTRAID_GITHUB_TOKEN"   # Add-ons -> Secrets, label EXACTLY this.
                                            # A read-only, single-repository fine-grained
                                            # PAT. Never typed into this notebook.
HF_SECRET       = "HF_TOKEN"      # optional; REQUIRED for the gated bases below

# ==================== THE BAKE-OFF =====================================
# Every entry is one arm. `enabled` decides whether this session runs it; results are
# written per base to out/results/<slug>.json and the comparison table at the end reads
# that directory, so a sweep can be SPLIT ACROSS SESSIONS (Kaggle caps a session at 12 h
# and a free account at 30 h a week).
#
# `licence` is a SELECTION CRITERION, not a footnote: this ships in a product.
# `approx_params` is what sits in GPU memory; `train_params` is the compute-relevant
#   (non-embedding) part, which is what the GPU-hour estimate scales with. For Gemma 4's
#   Per-Layer Embeddings the two differ by more than 2x, so both are reported.
# `lora_target_modules` is a FALLBACK. The notebook enumerates each checkpoint's real
#   nn.Linear leaves and prefers the intersection; a wrong list silently trains almost
#   nothing, so the trainable fraction is asserted into [0.05%, 5%] either way.
BASES = [
    {
        # ---------------------------------------------------- priority 1
        "slug": "gemma4-e2b", "hf_id": "google/gemma-4-E2B-it", "family": "gemma4",
        "approx_params": 5.12e9, "train_params": 2.3e9,
        "licence": "apache-2.0", "gated": False, "enabled": True,
        "per_device_bs": 2, "grad_accum": 16,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"],
        "dtype_notes":
            "TOP CANDIDATE on product grounds: Gemma 4 moved to Apache-2.0 (Gemma 3 was "
            "the restrictive Gemma Terms), it is the on-device tier Google built with "
            "Qualcomm and MediaTek, it underpins Gemini Nano 4 on Android, and the June "
            "2026 QAT release loads under 1 GB text-only in LiteRT-LM.\n"
            "        TWO THINGS TO WATCH HERE, BOTH MEASURED, NOT ASSUMED:\n"
            "        (1) 'E2B' is ~2.3 B EFFECTIVE via Per-Layer Embeddings but 5.12 B "
            "parameters resident -- ~10.3 GiB of fp16 weights on a 16 GiB card before a "
            "single activation. This arm runs at batch 2 x accum 16 for that reason and "
            "is the most likely to OOM.\n"
            "        (2) the text config carries `final_logit_softcapping`, and Gemma is "
            "the known fp16-overflow family. There is no bf16 on sm_60/sm_75 to hide it, "
            "so this arm watches the loss and aborts ITSELF rather than reporting a "
            "number produced by NaNs.",
        "gguf": "gemma4 arch, supported upstream. Multimodal checkpoint: the text model "
                "converts, the vision/audio towers are not needed and are not exported.",
    },
    {
        # ---------------------------------------------------- priority 2
        "slug": "granite4-nano-350m", "hf_id": "ibm-granite/granite-4.0-h-350m",
        "family": "granitemoehybrid", "approx_params": 0.34e9, "train_params": 0.34e9,
        "licence": "apache-2.0", "gated": False, "enabled": True,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "in_proj", "out_proj", "input_linear", "output_linear",
                                "gate_proj", "up_proj", "down_proj"],
        "dtype_notes":
            "HIGH VALUE, not a curiosity. This is the direct test of the hypothesis that "
            "~350 M is enough capacity for an ontology-constrained parse -- the shape "
            "Microsoft's Mu (330 M, encoder-decoder, Windows Settings) and the T5-small "
            "FRAME-parsing result both support (see the prior-art section of KAGGLE.md). "
            "If this arm lands close to the 0.8 B one, it is the shipping answer: 15x "
            "less resident memory than Gemma 4 E2B and the cheapest arm in the sweep to "
            "train. Apache-2.0, hybrid Mamba-2, natively supported in llama.cpp, vLLM and "
            "MLX.",
        "gguf": "Mamba-2 hybrid. IBM themselves ship a non-h twin for toolchains where "
                "Mamba-2 is not optimised (see the disabled entry below). If conversion "
                "fails, this arm still trains and scores through transformers and its "
                "GGUF column reads `unsupported`.",
    },
    {
        # ---------------------------------------------------- priority 3
        "slug": "qwen3.5-0.8b", "hf_id": "Qwen/Qwen3.5-0.8B", "family": "qwen3_5",
        "approx_params": 0.87e9, "train_params": 0.62e9,
        "licence": "apache-2.0", "gated": False, "enabled": True,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj",
                                "in_proj_qkvz", "in_proj_ba", "out_proj"],
        "dtype_notes":
            "The original pick and still valid. VERIFIED 2026-09-22: Qwen3.5 (2026-03-02) "
            "is STILL the newest Qwen with a small tier -- Qwen3.6 (35B-A3B, 27B) and "
            "Qwen3.8 (27B) ship nothing under 27 B, and both still report model_type "
            "`qwen3_5`, i.e. the same architecture.\n"
            "        Vision-language checkpoint (Qwen3_5ForConditionalGeneration): text "
            "stack only. Hybrid Gated DeltaNet at 3 linear layers : 1 full-attention "
            "layer, so most layers have NO q/k/v/o_proj. Vocabulary 248 320, so the "
            "output projection dominates memory. fp16 fine.",
        "gguf": "llama.cpp has upstream qwen3.5 / Gated-DeltaNet support; a build older "
                "than the Feb-2026 DeltaNet fixes loops. Build from master.",
    },
    {
        # ---------------------------------------------------- priority 4
        "slug": "lfm2.5-2.6b", "hf_id": "LiquidAI/LFM2.5-2.6B", "family": "lfm2",
        "approx_params": 2.70e9, "train_params": 2.70e9,
        "licence": "lfm1.0 (LFM Open License v1.0)", "gated": False, "enabled": True,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "out_proj",
                                "in_proj", "w1", "w2", "w3",
                                "gate_proj", "up_proj", "down_proj"],
        "dtype_notes":
            "Released 2026-08-04, 2.69 B, built for phones: day-one llama.cpp/GGUF, MLX "
            "and ONNX, vendor-reported ~30 tok/s on-device under 2.5 GB.\n"
            "        LICENCE CAVEAT, a future liability rather than a blocker: the LFM "
            "Open License v1.0 is Apache-derived but free commercial use ENDS above "
            "$10 M annual revenue. Somebody has to decide whether that is acceptable "
            "before this arm can win.\n"
            "        Also the largest arm in the sweep after Gemma: ~5.4 GiB resident and "
            "roughly 4x the 0.8 B's training cost.",
        "gguf": "first-party: `lfm2` is an upstream llama.cpp architecture.",
    },

    # ================= considered, not in the default sweep ==================
    # Flip `enabled` to add one. Each `dtype_notes` says why it is off.
    {
        "slug": "granite4-350m-dense", "hf_id": "ibm-granite/granite-4.0-350m",
        "family": "granitemoehybrid", "approx_params": 0.35e9, "train_params": 0.35e9,
        "licence": "apache-2.0", "gated": False, "enabled": False,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj",
                                "input_linear", "output_linear"],
        "dtype_notes": "IBM's non-hybrid twin of the Nano arm, published for exactly the "
                       "case where a toolchain's Mamba-2 support is not ready. Enable it "
                       "if granite4-nano-350m fails to convert or trains badly -- it "
                       "tests the same capacity hypothesis without the state-space risk.",
        "gguf": "conventional; converts with no special support.",
    },
    {
        "slug": "qwen3.5-2b", "hf_id": "Qwen/Qwen3.5-2B", "family": "qwen3_5",
        "approx_params": 2.27e9, "train_params": 1.9e9,
        "licence": "apache-2.0", "gated": False, "enabled": False,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj",
                                "in_proj_qkvz", "in_proj_ba", "out_proj"],
        "dtype_notes": "The 2 B arm of the same family: the within-family size-vs-accuracy "
                       "datapoint, ~3x the 0.8 B's training cost. Worth enabling once the "
                       "0.8 B number is known.",
        "gguf": "same as the 0.8 B.",
    },
    {
        "slug": "gemma4-e4b", "hf_id": "google/gemma-4-E4B-it", "family": "gemma4",
        "approx_params": 8.00e9, "train_params": 4.0e9,
        "licence": "apache-2.0", "gated": False, "enabled": False,
        "per_device_bs": 1, "grad_accum": 32,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"],
        "dtype_notes": "The next Gemma 4 on-device size. 8.0 B resident does not fit a "
                       "16 GiB card for training at all without sharding; listed so the "
                       "ceiling of the family is on the record.",
        "gguf": "gemma4 arch; supported upstream.",
    },
    # ---- dropped from consideration, and why (one line each) ----
    # smollm2-360m   HuggingFaceTB/SmolLM2-360M-Instruct -- superseded by Granite 4.0 Nano
    #                as the tiny-capacity probe, and it is a 2024-vintage model.
    # llama3.2-1b    meta-llama/Llama-3.2-1B-Instruct -- older generation (Sept 2024),
    #                gated, and the Llama 3.2 Community Licence carries a 700 M MAU clause.
    # phi4-mini      microsoft/Phi-4-mini-instruct -- MIT and strong, but 3.8 B is above
    #                the on-device target band.
    # tiny-aya       CC-BY-NC: non-commercial, so disqualified for a shipping product.
    # muse-glimmer   Meta Muse Glimmer 30B -- ~20 GiB at 4-bit, needs a 24 GiB GPU; far
    #                outside the mobile target. Not a candidate.
]

SWEEP_ORDER = "cheapest-first"   # "cheapest-first" | "as-listed"
                                 # A session is capped at 12 h. Running the cheap arms
                                 # first means an eviction costs the expensive arm, not
                                 # every arm. The comparison table keeps the priority
                                 # order regardless of the order they ran in.

# ---- the prompt -----------------------------------------------------
# "full" is Doctrine.swift verbatim (~700 tokens). See the note in the TRAINING SET
# section: at 45 722 rows x N epochs a 700-token fixed prefix costs ~7x the GPU time of
# the whole rest of the sequence, which does not fit a 12-hour Kaggle session.
SYSTEM_PROMPT_MODE = "none"      # "none" | "short" | "full"
MAX_SEQ_LEN     = 512            # raise to 1536 if SYSTEM_PROMPT_MODE == "full"

# ---- training (shared by every arm, so the comparison is fair) -------
SMOKE           = False   # True: 2 000 rows, 1 epoch, 80 eval rows, 40 decode turns.
                          # Prove the pipeline end to end in ~90 min before spending a
                          # session on the real run. KAGGLE.md says to do this first.
EPOCHS          = 2
LR              = 2.0e-4
WARMUP_RATIO    = 0.03
LORA_R          = 32
LORA_ALPHA      = 64
LORA_DROPOUT    = 0.05
PER_DEVICE_BS   = 8
GRAD_ACCUM      = 4
TRAIN_ROWS      = None     # None = every verified row
EVAL_N          = 600      # rows per validation file per epoch (both are scored in full
                           # at the final epoch)
SEED            = 11
MIN_TRAINABLE_PCT = 0.05   # a LoRA that adapts less than this is targeting the wrong
MAX_TRAINABLE_PCT = 5.0    # modules; more than this is not LoRA. Both abort the ARM.

# ---- export ---------------------------------------------------------
GGUF_QUANT      = "Q4_K_M"
EXPORT_EPOCH    = "best"   # "best" (highest distill_val exact) | "last" | an int epoch

# ---- final decode ---------------------------------------------------
LLAMA_CUDA      = True     # falls back to a CPU build, loudly, if the CUDA build fails
LLAMA_SLOTS     = 8        # llama-server parallel slots == decode concurrency
DECODE_MODES    = ["free", "teacher"]   # free is the headline; teacher is the diagnostic
GRAMMAR_MODE    = "gbnf"   # "gbnf" (convert here) | "json_schema" (let llama-server do it)
N_PREDICT       = 320

# ---- results --------------------------------------------------------
PUSH_RESULTS    = False    # LAST cell. Needs a token with WRITE scope. Off by default.
RESULTS_BRANCH  = "claude/qwen-mobile-results"

# ---- paths ----------------------------------------------------------
import os
WORK    = "/kaggle/working"                       # persisted, ~20 GB, survives eviction
SCRATCH = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp/centraid-scratch"
REPO    = os.path.join(SCRATCH, "centraid")       # re-clonable, so not in WORK
OUT     = os.path.join(WORK, "out")               # every artefact you download
RESULTS = os.path.join(OUT, "results")            # one JSON per base; the table reads it
CKPT    = os.path.join(WORK, "ckpt")              # adapters, one per base per epoch
LLAMA   = os.path.join(SCRATCH, "llama.cpp")
for _d in (SCRATCH, OUT, RESULTS, CKPT):
    os.makedirs(_d, exist_ok=True)

if SMOKE:
    TRAIN_ROWS, EPOCHS, EVAL_N = 2000, 1, 80

# The floor this run has to beat (Tier D, the hand-written parser, no model at all).
FLOOR = {"suite": (53, 120), "blind": (34, 60), "holdout": (31, 78)}

for _b in BASES:                       # per-arm overrides fall back to the shared knobs
    _b.setdefault("per_device_bs", PER_DEVICE_BS)
    _b.setdefault("grad_accum", GRAD_ACCUM)
    _b.setdefault("train_params", _b["approx_params"])
PRIORITY = {b["slug"]: i for i, b in enumerate(BASES)}
SWEEP = [b for b in BASES if b.get("enabled")]
if SWEEP_ORDER == "cheapest-first":
    SWEEP.sort(key=lambda b: b["train_params"])

PHASE_SECONDS = {}          # phase -> wall-clock seconds; the GPU budget report
import time, contextlib
@contextlib.contextmanager
def phase(name):
    t0 = time.time()
    print("\n### %s ..." % name, flush=True)
    try:
        yield
    finally:
        dt = time.time() - t0
        PHASE_SECONDS[name] = PHASE_SECONDS.get(name, 0.0) + dt
        print("### %s: %.1f s (%.2f h)" % (name, dt, dt / 3600.0), flush=True)

print("prompt mode %s | max seq %d | smoke %s | epochs %d | rows %s | order %s"
      % (SYSTEM_PROMPT_MODE, MAX_SEQ_LEN, SMOKE, EPOCHS, TRAIN_ROWS or "all",
         SWEEP_ORDER))

# ---- the budget, stated before anything is spent --------------------
# ~3.3 GPU-hours per billion COMPUTE-RELEVANT parameters per epoch, on a P100, for this
# corpus at SYSTEM_PROMPT_MODE='none'. Derived from ~8 x N FLOP per training token with
# gradient checkpointing, ~5 TFLOP/s realised, and ~5.9 M tokens an epoch.
GPU_H_PER_B_EPOCH = 3.3
OVERHEAD_H = 1.2            # rust build + llama.cpp build + selftest + corpus verify
def arm_hours(b):
    train = GPU_H_PER_B_EPOCH * (b["train_params"] / 1e9) * EPOCHS
    return train, train * 0.25 + 0.5      # (train, eval + export + decode + score)

print("\nsweep this session (%d arm(s)), cheapest first:" % len(SWEEP))
print("  %-20s %-34s %8s %8s %-26s %s"
      % ("slug", "hf_id", "resident", "compute", "licence", "est. GPU-h"))
_total = OVERHEAD_H
for b in SWEEP:
    t, e = arm_hours(b)
    _total += t + e
    print("  %-20s %-34s %6.2f B %6.2f B %-26s %5.1f  %s"
          % (b["slug"], b["hf_id"], b["approx_params"] / 1e9, b["train_params"] / 1e9,
             b["licence"], t + e, "GATED" if b["gated"] else ""))
print("  %-20s %-34s %8s %8s %-26s %5.1f" % ("", "+ shared builds", "", "", "", OVERHEAD_H))
print("  %-20s %-34s %8s %8s %-26s %5.1f" % ("TOTAL", "", "", "", "", _total))
print("\noff this session: %s"
      % ", ".join(b["slug"] for b in BASES if not b.get("enabled")))

print("\nKaggle free tier: a SESSION is capped at 12 h and an ACCOUNT at 30 h a week.")
if _total > 11.0:
    print("*** %.1f GPU-hours does NOT fit one session. Results are written per arm to\n"
          "    out/results/ as each finishes, so an eviction costs the arm in flight and\n"
          "    nothing else -- but plan the split. Suggested, at EPOCHS=%d:\n"
          "      session 1  granite4-nano-350m + qwen3.5-0.8b        ~%.0f h\n"
          "      session 2  gemma4-e2b                               ~%.0f h\n"
          "      session 3  lfm2.5-2.6b                              ~%.0f h\n"
          "    Set `enabled` accordingly and re-run; the comparison table at the end reads\n"
          "    out/results/ and assembles whatever is there, across sessions.\n"
          "    Cheaper levers, in the order they cost you least: EPOCHS=1 (halves it),\n"
          "    TRAIN_ROWS=20000 (halves it again), drop an arm."
          % (_total, EPOCHS,
             OVERHEAD_H + sum(sum(arm_hours(b)) for b in BASES
                              if b["slug"] in ("granite4-nano-350m", "qwen3.5-0.8b")),
             OVERHEAD_H + sum(sum(arm_hours(b)) for b in BASES
                              if b["slug"] == "gemma4-e2b"),
             OVERHEAD_H + sum(sum(arm_hours(b)) for b in BASES
                              if b["slug"] == "lfm2.5-2.6b")))
else:
    print("    %.1f GPU-hours: fits one session." % _total)


## Phase 0 — the machine

Kaggle's free accelerators are **P100 (sm_60)** and **2 x T4 (sm_75)**. Neither supports
bfloat16 and neither supports FlashAttention-2 (which needs sm_80). Everything here runs in
**fp16** with SDPA attention.

The one failure this cell exists to catch: **CUDA 13 dropped Pascal**. A PyTorch wheel built
against CUDA 13 has no `sm_60` in its arch list, so on a P100 it will either refuse to launch
kernels or silently produce garbage. The cell aborts with the fix rather than training for
six hours into NaNs.


In [ ]:
import sys, subprocess, torch

class Abort(RuntimeError):
    """Something is wrong with the run itself. Never caught, never worked around."""

print("torch %s   cuda %s" % (torch.__version__, torch.version.cuda))
if not torch.cuda.is_available():
    raise Abort("No GPU. Kaggle: Notebook options -> Accelerator -> GPU P100 (or T4 x2).")

DEV_NAME = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
SM = "sm_%d%d" % CAP
ARCHES = torch.cuda.get_arch_list()
TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("device     %s  %s  %.1f GiB  (%d visible)"
      % (DEV_NAME, SM, TOTAL_GB, torch.cuda.device_count()))
print("torch built for: %s" % " ".join(ARCHES))

if SM not in ARCHES:
    raise Abort(
        "This torch build has no %s kernels (it was built for: %s).\n"
        "CUDA 13 dropped Maxwell/Pascal/Volta, so a cu13 wheel cannot drive a P100.\n"
        "Fix, in order of preference:\n"
        "  1. Notebook options -> Accelerator -> **T4 x2** (sm_75, still supported), or\n"
        "  2. pin an older wheel in the deps cell:\n"
        "     pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121\n"
        "Do NOT continue: the run would produce NaNs or silently wrong numbers."
        % (SM, " ".join(ARCHES)))

# bf16 and FlashAttention-2 both need sm_80. State it, do not discover it later.
BF16_OK = CAP >= (8, 0) and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
ATTN_IMPL = "sdpa"
try:
    import flash_attn  # noqa: F401
    if CAP >= (8, 0):
        ATTN_IMPL = "flash_attention_2"
except Exception:
    pass
print("dtype      %s   attn %s   (bf16 supported: %s)"
      % (str(DTYPE).split(".")[-1], ATTN_IMPL, BF16_OK))
if not BF16_OK:
    print("           -> fp16 with a GradScaler. bf16 is NOT available on %s." % SM)

# A real kernel, in the dtype we will train in. If this is not finite, nothing else matters.
_a = torch.randn(512, 512, device="cuda", dtype=DTYPE)
_b = torch.randn(512, 512, device="cuda", dtype=DTYPE)
_c = (_a @ _b).float()
if not torch.isfinite(_c).all():
    raise Abort("A %s matmul on %s produced non-finite values. Stop: the arithmetic is "
                "broken, every number after this would be noise." % (DTYPE, SM))
print("fp check   %s matmul finite, mean |x| = %.3f" % (DTYPE, _c.abs().mean().item()))
del _a, _b, _c
torch.cuda.empty_cache()

print("\ndisk:")
subprocess.run(["df", "-h", "/kaggle/working", SCRATCH], check=False)


In [ ]:
import os, shutil, subprocess, textwrap

# ---------------------------------------------------------------- the clone
# The repository is private. The token comes from a Kaggle Secret and is never written
# into a cell, never printed, and is scrubbed out of .git/config immediately after the
# clone so it cannot leak through a notebook output or a saved version.
def _clone():
    from kaggle_secrets import UserSecretsClient
    try:
        token = UserSecretsClient().get_secret(KAGGLE_SECRET)
    except Exception as exc:
        raise Abort(
            "Could not read the Kaggle Secret %r (%s).\n"
            "Add-ons -> Secrets -> Add a secret, LABEL exactly %r, value = a fine-grained\n"
            "GitHub PAT scoped to this ONE repository with Contents: Read-only. Then tick\n"
            "'Attach to notebook'. Also check Notebook options -> Internet is ON."
            % (KAGGLE_SECRET, type(exc).__name__, KAGGLE_SECRET)) from None
    if not token or len(token) < 20:
        raise Abort("The secret %r is empty or too short to be a PAT." % KAGGLE_SECRET)
    if os.path.isdir(REPO):
        shutil.rmtree(REPO)
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS="/bin/true")
    url = "https://x-access-token:%s@%s" % (token, GIT_HOST)
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch", GIT_BRANCH,
         url, REPO],
        env=env, capture_output=True, text=True)
    if proc.returncode != 0:
        # scrub the token out of anything git says before it reaches the output
        msg = (proc.stderr or "").replace(token, "<redacted>")
        raise Abort("git clone failed (rc=%d):\n%s\n"
                    "Check: the PAT has Contents:Read on %s, it has not expired, and the\n"
                    "branch %r exists." % (proc.returncode, msg, GIT_HOST, GIT_BRANCH))
    # the token is in .git/config after a clone -- remove it now.
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://" + GIT_HOST], check=True)
    del token

with phase("clone"):
    _clone()

head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
print("HEAD      ", head)
if head != GIT_COMMIT:
    msg = ("HEAD is %s, this notebook was written against %s.\n"
           "The branch moved. Re-read the paths this notebook depends on (below) before\n"
           "trusting a number out of it, or set ALLOW_COMMIT_DRIFT = True to proceed."
           % (head, GIT_COMMIT))
    if ALLOW_COMMIT_DRIFT:
        print("WARNING: " + msg)
    else:
        raise Abort(msg)

# ------------------------------------------------- every path this notebook needs
AFM      = os.path.join(REPO, "experiments", "afm-spike")
CANON    = os.path.join(REPO, "experiments", "canon-model")
DATA     = os.path.join(CANON, "data")
GRAMMAR  = os.path.join(REPO, "crates", "evalsuite", "grammar")
NEEDED = {
    "frame.py (the frame, both directions)":  os.path.join(AFM, "frame.py"),
    "render_frames.py (frame -> canonical)":  os.path.join(AFM, "render_frames.py"),
    "gen_vocab.py (the doctrine text)":       os.path.join(AFM, "gen_vocab.py"),
    "summarise.py":                           os.path.join(AFM, "summarise.py"),
    "Makefile (selftest)":                    os.path.join(AFM, "Makefile"),
    "check.py (the grammar parser)":          os.path.join(GRAMMAR, "check.py"),
    "lexicon.py (the vocabularies)":          os.path.join(GRAMMAR, "lexicon.py"),
    "map.json (434 gold turns)":              os.path.join(GRAMMAR, "map.json"),
    "render.py (the unparser)":               os.path.join(CANON, "encoder", "render.py"),
    "distill.jsonl (the training corpus)":    os.path.join(DATA, "distill.jsonl"),
    "distill_val.jsonl (held out by skeleton)": os.path.join(DATA, "distill_val.jsonl"),
    "val.jsonl (held-out templates)":         os.path.join(DATA, "val.jsonl"),
    "run-model.rs (the scorer)":              os.path.join(
        REPO, "crates", "candidates", "src", "bin", "run-model.rs"),
    "overlap-check.rs (the leakage gate)":    os.path.join(
        REPO, "crates", "evalsuite", "src", "bin", "overlap-check.rs"),
    "suite.json":   os.path.join(REPO, "crates", "evalsuite", "suite.json"),
    "blind.json":   os.path.join(REPO, "crates", "evalsuite", "blind.json"),
    "holdout.json": os.path.join(REPO, "crates", "evalsuite", "holdout.json"),
}
missing = [(k, v) for k, v in NEEDED.items() if not os.path.exists(v)]
if missing:
    raise Abort("These paths are gone from the branch:\n" +
                "\n".join("  %-42s %s" % (k, v) for k, v in missing))
for k, v in NEEDED.items():
    print("  ok  %-44s %8.1f KiB" % (k, os.path.getsize(v) / 1024.0))

# frame.py and render_frames.py are imported directly: this notebook IS the harness.
for p in (AFM, GRAMMAR, os.path.join(CANON, "encoder")):
    if p not in sys.path:
        sys.path.insert(0, p)


In [ ]:
import json, subprocess, sys, os

def pip(*args):
    rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode
    if rc != 0:
        raise Abort("pip install %s failed (rc=%d)" % (" ".join(args), rc))

with phase("python deps"):
    pip("-U", "pip")
    pip("-U", "transformers", "peft", "accelerate", "datasets", "huggingface_hub",
        "hf_transfer", "jsonschema", "sentencepiece", "protobuf", "requests")
    # llguidance gives a JSON-schema-constrained decode inside transformers, for any base
    # whose architecture llama.cpp cannot convert. Optional: if it will not install, that
    # arm decodes UNCONSTRAINED and the result says so rather than pretending.
    try:
        pip("-U", "llguidance")
        HAVE_LLGUIDANCE = True
    except Abort:
        HAVE_LLGUIDANCE = False
        print("llguidance did not install; the transformers fallback decode will be "
              "unconstrained and will be labelled as such.")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

HAVE_HF_TOKEN = False
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret(HF_SECRET)
    HAVE_HF_TOKEN = True
    print("HF token secret %r found." % HF_SECRET)
except Exception:
    print("No %r secret. Ungated Apache/MIT bases are fine; GATED bases will be SKIPPED."
          % HF_SECRET)

# ------------------------------------------------- probe every arm's architecture
# In a SUBPROCESS, so the kernel imports transformers exactly once, after the right
# version is installed. A base whose architecture this transformers does not know is not
# an error for the sweep: it is one arm, reported as unsupported.
PROBE = r"""
import json, sys, transformers
from transformers import AutoConfig
out = {"transformers": transformers.__version__, "models": {}}
for mid in json.loads(sys.argv[1]):
    try:
        c = AutoConfig.from_pretrained(mid)
        t = getattr(c, "text_config", None) or c
        out["models"][mid] = {
            "ok": True, "model_type": c.model_type,
            "architectures": list(getattr(c, "architectures", None) or []),
            "hidden_size": getattr(t, "hidden_size", None),
            "num_hidden_layers": getattr(t, "num_hidden_layers", None),
            "vocab_size": getattr(t, "vocab_size", None),
            "layer_types": sorted(set(getattr(t, "layer_types", []) or [])),
            "has_vision": bool(getattr(c, "vision_config", None)),
        }
    except Exception as exc:
        out["models"][mid] = {"ok": False, "why": type(exc).__name__ + ": " + str(exc)[:300]}
print("PROBE " + json.dumps(out))
"""

def probe(ids):
    proc = subprocess.run([sys.executable, "-c", PROBE, json.dumps(ids)],
                          capture_output=True, text=True)
    for line in proc.stdout.splitlines():
        if line.startswith("PROBE "):
            return json.loads(line[6:])
    raise Abort("the architecture probe printed nothing:\n" + (proc.stderr or "")[-2000:])

IDS = [b["hf_id"] for b in SWEEP]
info = probe(IDS)
if any(not v["ok"] for v in info["models"].values()):
    print("transformers %s does not know: %s\ninstalling transformers from git main"
          % (info["transformers"],
             ", ".join(k for k, v in info["models"].items() if not v["ok"])))
    with phase("transformers from source"):
        pip("-U", "git+https://github.com/huggingface/transformers")
    info = probe(IDS)

print("\ntransformers %s" % info["transformers"])
RUNNABLE = []
for base in SWEEP:
    meta = info["models"][base["hf_id"]]
    base["probe"] = meta
    if not meta["ok"]:
        print("  SKIP  %-14s architecture unknown to transformers: %s"
              % (base["slug"], meta["why"]))
        base["skip"] = "transformers cannot load this architecture: " + meta["why"]
        continue
    if base["gated"] and not HAVE_HF_TOKEN:
        print("  SKIP  %-14s gated repo (%s licence) and no %s secret. Accept the terms\n"
              "        on huggingface.co with the account the token belongs to, then add\n"
              "        the token as a Kaggle Secret named %r."
              % (base["slug"], base["licence"], HF_SECRET, HF_SECRET))
        base["skip"] = "gated repo, no HF token"
        continue
    hybrid = [t for t in meta["layer_types"] if t != "full_attention"]
    print("  run   %-14s %-16s layers %-3s hidden %-5s vocab %-7s %s%s"
          % (base["slug"], meta["model_type"], meta["num_hidden_layers"],
             meta["hidden_size"], meta["vocab_size"],
             "VL " if meta["has_vision"] else "",
             ("hybrid: " + ",".join(hybrid)) if hybrid else "dense"))
    RUNNABLE.append(base)

if not RUNNABLE:
    raise Abort("Every arm in SWEEP was skipped. Nothing to do.")
print("\n%d arm(s) will run." % len(RUNNABLE))


In [ ]:
import os, re, subprocess, shutil

# The scorer is Rust and is needed TWICE: once to prove the wiring before training
# (`make selftest`) and once to score the model at the end. The leakage gate is a second
# binary from the same workspace. Both are built here, once.
CARGO_HOME = os.path.join(SCRATCH, "cargo")
RUSTUP_HOME = os.path.join(SCRATCH, "rustup")
TARGET_DIR = os.path.join(SCRATCH, "target")     # several GiB; keep it out of /kaggle/working
os.environ.update(CARGO_HOME=CARGO_HOME, RUSTUP_HOME=RUSTUP_HOME,
                  CARGO_TARGET_DIR=TARGET_DIR,
                  CARGO_NET_RETRY="5", CARGO_TERM_COLOR="never",
                  RUSTFLAGS=os.environ.get("RUSTFLAGS", ""))
os.environ["PATH"] = os.path.join(CARGO_HOME, "bin") + ":" + os.environ["PATH"]

def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    proc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str),
                          capture_output=True, text=True,
                          env=dict(os.environ, **(env or {})))
    if not quiet:
        if proc.stdout:
            print(proc.stdout[-8000:])
        if proc.stderr:
            print(proc.stderr[-4000:])
    if check and proc.returncode != 0:
        raise Abort("command failed (rc=%d): %s" % (proc.returncode, cmd))
    return proc

with phase("rust toolchain"):
    if not shutil.which("cargo"):
        sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs "
           "| sh -s -- -y --profile minimal --default-toolchain none --no-modify-path")
    # rust-toolchain.toml pins the channel; running cargo inside the repo installs it.
    sh(["cargo", "--version"], cwd=REPO)

with phase("cargo build run-model + overlap-check"):
    sh(["cargo", "build", "--release", "-p", "centraid-candidates", "--bin", "run-model"],
       cwd=REPO)
    sh(["cargo", "build", "--release", "-p", "centraid-evalsuite", "--bin",
        "overlap-check"], cwd=REPO)
RUN_MODEL = os.path.join(TARGET_DIR, "release", "run-model")
OVERLAP = os.path.join(TARGET_DIR, "release", "overlap-check")
for p in (RUN_MODEL, OVERLAP):
    if not os.path.exists(p):
        raise Abort("cargo said it built, but %s is not there." % p)
print("built:", RUN_MODEL, "(%.1f MiB)" % (os.path.getsize(RUN_MODEL) / 1024**2))
print("built:", OVERLAP)

# ------------------------------------------------------------------ the scorer, parsed
_STRICT = re.compile(r"SESSIONS PASSED \(strict, the headline\):\s+(\d+)/(\d+)")
_GRADED = re.compile(r"GRADED SESSION SCORE \(F1 per rows turn\):\s+([\d.]+)%")
_TURNS = re.compile(r"TURNS PASSED \(diagnostic, never a score\):\s+(\d+)/(\d+)")
_CORPUS = re.compile(r"=+ corpus: (\w+) =+")

def score_with_run_model(outputs_path, corpus="all"):
    """The repository's own executor, over a file written before anything parsed it."""
    proc = subprocess.run([RUN_MODEL, "--corpus", corpus, "--outputs",
                           os.path.abspath(outputs_path)],
                          cwd=REPO, capture_output=True, text=True)
    text = proc.stdout + proc.stderr
    print(text[-12000:])
    if proc.returncode != 0:
        raise Abort("run-model failed (rc=%d) on %s" % (proc.returncode, outputs_path))
    out, name = {}, None
    for line in text.splitlines():
        m = _CORPUS.search(line)
        if m:
            name = m.group(1)
            out[name] = {}
            continue
        if name is None:
            continue
        m = _STRICT.search(line)
        if m:
            out[name]["strict"] = (int(m.group(1)), int(m.group(2)))
        m = _GRADED.search(line)
        if m:
            out[name]["graded"] = float(m.group(1))
        m = _TURNS.search(line)
        if m:
            out[name]["turns"] = (int(m.group(1)), int(m.group(2)))
    if not out:
        raise Abort("run-model printed no corpus report. Its output format changed; the\n"
                    "regexes in this cell are the thing to fix, not the scorer.")
    return out


## Phase 1 — prove the wiring before the model exists

`python3 frame.py` proves `parse(gold) -> encode -> flat -> decode -> render` is
**tree-equal** to `parse(gold)` over all 434 gold canonicals. That number is the ceiling of
this whole experiment.

`make selftest` then stands **gold frames in for the model's** and pushes them through the
real renderer and the real scorer. It must print **120/120 · 60/60 · 78/78**. Until it
does, a red number later could be the harness rather than the model; after it does, a red
number is a finding about the model.

Do not continue past a red cell here. Do not "fix" it by changing the expectation.


In [ ]:
import frame, check, render_frames, gen_vocab   # the repository's own modules

with phase("frame coverage"):
    proc = sh([sys.executable, "frame.py"], cwd=AFM, check=False)
cov = re.findall(r"^(gold, .+?)\s+(\d+)/(\d+)", proc.stdout, re.M)
if not cov:
    raise Abort("frame.py printed no coverage lines:\n" + proc.stdout[-2000:])
for label, ok, total in cov:
    print("  %-24s %s/%s" % (label.strip(), ok, total))
    if ok != total:
        raise Abort("frame.py covers only %s/%s of the gold canonicals (%s). The frame "
                    "cannot express the corpus; a model score would be capped below the "
                    "floor by the HARNESS. Stop." % (ok, total, label.strip()))

with phase("make selftest (gold frames through the real scorer)"):
    proc = sh(["make", "selftest"], cwd=AFM, check=False)
text = proc.stdout + proc.stderr
m = re.search(r"selftest: (\d+)/(\d+) tree-equal to gold", text)
if not m or m.group(1) != m.group(2):
    raise Abort("render_frames --selftest is not tree-equal to gold:\n" + text[-4000:])
print("  selftest tree-equal: %s/%s" % m.groups())

got, name = {}, None
for line in text.splitlines():
    mm = _CORPUS.search(line)
    if mm:
        name = mm.group(1)
    mm = _STRICT.search(line)
    if mm and name:
        got[name] = (int(mm.group(1)), int(mm.group(2)))
EXPECT = {"suite": (120, 120), "blind": (60, 60), "holdout": (78, 78)}
print("\n  corpus    gold frames through run-model   expected")
for corpus, want in EXPECT.items():
    have = got.get(corpus)
    print("  %-9s %-32s %s" % (corpus, "%s/%s" % have if have else "MISSING",
                               "%d/%d" % want))
if got != EXPECT:
    raise Abort(
        "The ORACLE path does not reproduce %s.\n"
        "Gold frames are standing in for the model here, so this is the harness, not the\n"
        "model. Send the output back rather than continuing: every score below would be\n"
        "measured against a broken renderer." % EXPECT)
print("\nWIRING PROVEN. Anything red from here on is a finding about the model.")


## Phase 2 — the training set, and what verifying it can and cannot measure

`distill.jsonl` is 45 779 rows of `"<previous canonical> ||| <utterance>" -> <canonical>`.
Each canonical is turned into the **flat frame** the model has to emit, by
`frame.to_flat(frame.encode(check.parse(target)))` — the same call the self-test above used
to stand in for the model. Every row is then run back the other way through the *real
renderer*, `render_frames.render_one`, and kept only if the canonical it rebuilds is
**tree-equal** to the row's own gold.

### What this gate measures — read this before quoting its number

The brief for this lane asked for the drop rate to be reported as "the previously-unmeasured
error rate of the distillation generator". **It is not that, and it cannot be.** Three
things are true and worth separating:

1. **The round-trip gate is nearly empty.** Every distill target is a canonical that already
   came out of `train.jsonl`, and `frame.py` covers those at 100%; the verbatim rule was
   already applied by `distill_assemble.py`. Measured on this branch, the gate drops **1 row
   in 45 779** (0.002%) from `distill.jsonl` and **0** from `distill_val.jsonl`. A drop rate
   of 0.002% is not a generator error rate; reporting it as one would be false.
2. **There is no execution check available for these rows.** `run-model` scores model output
   against a fixed corpus (`suite`/`blind`/`holdout`) over a real vault. `distill.jsonl`'s
   canonicals name an *invented second world* (`distill_world.py`) whose handles exist in no
   vault, so executing them would return nothing for essentially every row — no signal.
   The Rust scorer is used where it has meaning: the wiring proof above, and the final score.
3. **A second gate does find something.** Ask instead: *is every literal the target carries
   recoverable from what the person said?* Split by class — **handles** (`called`, and the
   content-bearing write arguments) against **keywords** (`status: "completed"` for "mark it
   done", `currency in ("GBP")` for "in pounds"). A row whose *handle* appears nowhere in the
   sentence or the previous canonical is a row the model cannot get right by reading, and
   several of them are simply mispaired. Keyword absence is a lexical mapping the model must
   learn and is **not** dropped.

Measured on this branch, that second gate is where the corpora separate:

| corpus | rows | round-trip drop | + unrecoverable handle | kept |
| --- | --- | --- | --- | --- |
| `distill.jsonl` | 45 779 | 1 (0.002%) | 56 (0.12%) | **45 722 (99.88%)** |
| `distill_val.jsonl` | 2 159 | 0 | 2 (0.09%) | 2 157 (99.91%) |
| `val.jsonl` (templates) | 2 213 | 30 (1.36%) | 36 (1.63%) | 2 147 (97.02%) |
| `train.jsonl` (templates) | 14 043 | 243 (1.73%, **20 do not parse at all**) | 279 (1.99%) | 13 521 (96.28%) |

**The finding is the comparison, not the absolute.** The old template corpus is ~16x more
likely than the distilled corpus to hand a model a target whose handle the sentence never
says — i.e. to *train it to invent a name*. That is a concrete, previously-unquantified
mechanism behind the training lane's observation that no model output ever copied a world
handle out of a request.

**What stays unmeasured**: whether each sentence actually *means* its canonical. That is a
semantic judgement, `DISTILL.md` says so plainly, and no mechanical gate in this repository
settles it. It is the corpus's real error bar and this notebook does not close it.


In [ ]:
import collections, json, os, random

# Handle-class arguments: a value that NAMES something in the vault. Everything else
# (status, currency, kind, favorite ...) is a keyword the model must learn to map onto.
HANDLE_ARGS = set(render_frames.CONTENT_ARGS) | {"notebook", "album", "folder"}

def _literals(flat):
    """(class, where, value) for every string literal a flat frame carries."""
    out = []
    def walk(s, where):
        if not isinstance(s, dict):
            return
        for key in ("called", "called2"):
            if s.get(key):
                out.append(("handle", where + "." + key, s[key]))
        for key in ("filtersA", "filtersB", "filtersC"):
            pred = s.get(key)
            if not pred:
                continue
            for atom in pred.get("atoms") or []:
                if atom.get("valueKind") == "literal" and isinstance(atom.get("value"), str):
                    out.append(("keyword", where + ".filter", atom["value"]))
                for lit in atom.get("lits") or []:
                    if isinstance(lit, str):
                        out.append(("keyword", where + ".oneof", lit))
                if atom.get("set"):
                    walk(atom["set"], where + ".atomset")
        if s.get("combineRight"):
            walk(s["combineRight"], where + ".right")
    if flat.get("set"):
        walk(flat["set"], "set")
    if flat.get("set2"):
        walk(flat["set2"], "set2")
    for cmd in flat.get("cmds") or []:
        for arg in cmd.get("args") or []:
            name = str(arg.get("name"))
            if (arg.get("valueKind") or "literal") == "literal" and \
                    isinstance(arg.get("value"), str):
                out.append(("handle" if name in HANDLE_ARGS else "keyword",
                            "arg." + name, arg["value"]))
            if arg.get("set"):
                walk(arg["set"], "arg.on")
        if cmd.get("on"):
            walk(cmd["on"], "cmd.on")
    return out

def verify_corpus(path, label, keep=True):
    """Round-trip through the real renderer, then handle-recoverability. Reports both."""
    kept, stats = [], collections.Counter()
    hl_tot = hl_abs = kw_tot = kw_abs = 0
    absent_sites = collections.Counter()
    examples = []
    n = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            n += 1
            target = row["target"]
            prev, _, utterance = row["input"].partition(" ||| ")
            prev = "" if prev == "NONE" else prev
            try:
                gold = check.parse(target)
            except Exception:
                stats["target does not parse"] += 1
                continue
            try:
                flat = json.loads(json.dumps(frame.to_flat(frame.encode(gold))))
            except Exception as exc:
                stats["not expressible as a frame (%s)" % type(exc).__name__] += 1
                continue
            haystack = "%s %s" % (prev, utterance)
            canonical, reason = render_frames.render_one(json.dumps(flat), haystack)
            if not canonical:
                stats["renderer refused: " + reason.split(":")[0]] += 1
                continue
            if check.parse(canonical) != gold:
                stats["round-trip is not tree-equal"] += 1
                continue
            said = render_frames.words(haystack)
            unrecoverable = False
            for klass, where, value in _literals(flat):
                w = render_frames.words(value)
                if not w:
                    continue
                if klass == "handle":
                    hl_tot += 1
                    if not (w & said):
                        hl_abs += 1
                        unrecoverable = True
                        absent_sites[where] += 1
                        if len(examples) < 8:
                            examples.append((where, value, utterance[:80], target[:80]))
                else:
                    kw_tot += 1
                    if not (w & said):
                        kw_abs += 1
            if unrecoverable:
                stats["handle the sentence never says"] += 1
                continue
            if keep:
                kept.append({"prev": prev, "request": utterance, "flat": flat,
                             "target": target,
                             "template_id": row.get("template_id", ""),
                             "move": row.get("move", ""),
                             "head": flat["head"]})
    dropped = sum(stats.values())
    print("\n%-22s rows %6d   verified %6d (%.3f%%)   dropped %d (%.3f%%)"
          % (label, n, n - dropped, 100.0 * (n - dropped) / n, dropped,
             100.0 * dropped / n))
    for why, count in stats.most_common():
        print("      %-44s %d" % (why, count))
    print("      handle literals %6d, absent %4d (%.2f%%) | keyword literals %5d, "
          "absent %4d (%.2f%%)  [keyword absence is NOT a drop]"
          % (hl_tot, hl_abs, 100.0 * hl_abs / max(1, hl_tot),
             kw_tot, kw_abs, 100.0 * kw_abs / max(1, kw_tot)))
    if absent_sites:
        print("      unrecoverable handle sites:", dict(absent_sites.most_common(6)))
    for where, value, utterance, target in examples[:4]:
        print("      e.g. %-16s %r\n           said: %s\n           gold: %s"
              % (where, value, utterance, target))
    return kept, {"rows": n, "kept": n - dropped, "dropped": dropped,
                  "why": dict(stats), "handle_literals": hl_tot,
                  "handle_absent": hl_abs, "keyword_literals": kw_tot,
                  "keyword_absent": kw_abs}

with phase("verify the corpora"):
    TRAIN, TRAIN_STATS = verify_corpus(os.path.join(DATA, "distill.jsonl"),
                                       "distill.jsonl")
    VAL_DISTILL, VD_STATS = verify_corpus(os.path.join(DATA, "distill_val.jsonl"),
                                          "distill_val.jsonl")
    VAL_TEMPLATE, VT_STATS = verify_corpus(os.path.join(DATA, "val.jsonl"),
                                           "val.jsonl (templates)")
    # the old corpus, measured but never trained on -- the comparison IS the finding
    _, OLD_STATS = verify_corpus(os.path.join(DATA, "train.jsonl"),
                                 "train.jsonl (templates, NOT trained on)", keep=False)

VERIFY_REPORT = {"distill": TRAIN_STATS, "distill_val": VD_STATS,
                 "val_template": VT_STATS, "train_template_baseline": OLD_STATS}
with open(os.path.join(OUT, "verify.json"), "w") as fh:
    json.dump(VERIFY_REPORT, fh, indent=2)

print("\n" + "=" * 78)
print("DROP RATE, stated plainly:")
for name, st in VERIFY_REPORT.items():
    print("  %-26s %6d rows, dropped %4d (%.3f%%)"
          % (name, st["rows"], st["dropped"], 100.0 * st["dropped"] / st["rows"]))
print("This is a FRAME-EXPRESSIBILITY and HANDLE-RECOVERABILITY rate. It is NOT the\n"
      "distillation generator's semantic error rate, which stays unmeasured -- see the\n"
      "markdown above. The number worth carrying forward is the ratio between the\n"
      "distilled corpus and the template corpus, not either one alone.")
print("=" * 78)

random.Random(SEED).shuffle(TRAIN)
UNVERIFIED_SIZE = TRAIN_STATS["rows"]
if TRAIN_ROWS:
    TRAIN = TRAIN[:TRAIN_ROWS]
print("\ntraining on %d verified rows (unverified corpus was %d)"
      % (len(TRAIN), UNVERIFIED_SIZE))
print("by head:", dict(collections.Counter(r["head"] for r in TRAIN).most_common()))
print("by move:", dict(collections.Counter(r["move"] for r in TRAIN).most_common()))
ABSTAIN = {"refuse", "nothing"}
n_abstain = sum(1 for r in TRAIN if r["head"] in ABSTAIN)
print("abstention rows (refuse/nothing): %d (%.2f%%) -- scored as their own row, never "
      "pooled" % (n_abstain, 100.0 * n_abstain / len(TRAIN)))


In [ ]:
import json, os, subprocess

# The gate is the repository's own binary, run over the rows this notebook will actually
# train on -- not over the file as shipped, because the verification above changed the set.
# `overlap-check` reads a `request` field (and `template_id`), so the verified rows are
# materialised in that shape. It FAILS the run on an exact request collision with
# suite/blind/holdout, or on a template id shared with the validation file.
def as_overlap_rows(rows, path):
    with open(path, "w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps({"request": row["request"],
                                 "template_id": row["template_id"]}) + "\n")
    return path

GATE_TRAIN = as_overlap_rows(TRAIN, os.path.join(OUT, "_gate-train.jsonl"))
GATE_VAL = as_overlap_rows(VAL_DISTILL, os.path.join(OUT, "_gate-val.jsonl"))

with phase("leakage gate"):
    proc = subprocess.run([OVERLAP, GATE_TRAIN, GATE_VAL],
                          cwd=REPO, capture_output=True, text=True)
    print(proc.stdout[-14000:])
    if proc.stderr:
        print(proc.stderr[-3000:])

if proc.returncode != 0:
    raise Abort(
        "THE LEAKAGE GATE TRIPPED (rc=%d).\n"
        "An exact request collision with suite/blind/holdout, or a template id shared\n"
        "between the training rows and the validation rows. Either way every score this\n"
        "notebook would print afterwards measures memorisation and reports it as\n"
        "competence -- and the scores go UP, so it is invisible.\n"
        "Do NOT relax the gate, drop the corpus argument, or filter the report. The fix\n"
        "belongs to whoever owns the data: take the output above to the corpus lane.\n"
        % proc.returncode)

# Report the 4-gram rate, which the gate prints but never fails on: it is a RATE, and the
# rate is what tells ordinary English apart from a paraphrase of the corpus. DISTILL.md's
# published baseline for distill.jsonl is 29/248 (11.7%) on suite and 5/72 (6.9%) on blind,
# against train.jsonl's 38/248 and 6/72.
import re as _re
for line in proc.stdout.splitlines():
    if _re.search(r"(four|4)-gram|exact|proper", line, _re.I):
        print("  " + line.strip())
print("\nLEAKAGE GATE PASSED over the %d rows this run trains on." % len(TRAIN))


## The grammar, derived from `frame.py`

The final decode is **constrained**: the model may only emit a JSON object the flat frame
can actually hold. The schema is built from `frame.py`'s own module constants
(`FLAT_HEADS`, `ATOM_TYPES`, `CMPS`, `VALUE_KINDS`, `SLOT_ORDER`) and the grammar's
`lexicon.py` (kinds, fields, refs, window phrases, decline reasons, verbs). Nothing is
typed out by hand, so the grammar cannot drift from the renderer that has to accept its
output.

**The schema is proved before it is used**: every one of the 434 gold wire frames must
validate against it. A schema that rejects a gold frame would cap the score below the floor
for a reason that has nothing to do with the model, so a failure here aborts.

One finding carried in this cell: `frame.VALUE_KINDS` is **dead and wrong** — nothing in
`frame.py` reads it, and it omits `"value"`, which `_rhs_to_flat` emits and
`_rhs_from_flat` accepts for an argument the vault computes (`due_at: dtstart of (it)`).
Two gold turns (suite s44 t1, s74 t0) use it. The schema takes the union.


In [ ]:
import importlib.util, json, os
import lexicon

def _ref(name):
    return {"$ref": "#/$defs/" + name}

def frame_json_schema():
    """The flat frame's JSON Schema. Every vocabulary is READ from the repository."""
    KINDS = sorted(lexicon.KINDS)
    FIELDS = sorted(lexicon.FIELDS)
    REFS = sorted(lexicon.REFS) + ["ordinal"]
    REASONS = sorted(lexicon.DECLINE_REASONS)
    VERBS = sorted(set(lexicon.COMMANDS) | set(lexicon.VERB_CLASSES))
    # `frame.VALUE_KINDS` is dead and omits "value" -- see the markdown above.
    VALUE_KINDS = sorted(set(frame.VALUE_KINDS) | {"value"})
    # An argument NAME is not an enum: check.py's cmd() reads it with word(), and the verb
    # classes use names ("to", "by") that are in no registry. Carry the ident pattern.
    ARG_NAME = {"type": "string", "pattern": r"^[A-Za-z_][A-Za-z0-9_.]*$"}

    window = {"type": "object", "additionalProperties": False, "required": ["how"],
              "properties": {
                  "how": {"enum": ["phrase", "date", "datetime", "month", "daterange",
                                   "rolling", "anchored"]},
                  "value": {"type": "string"},
                  "n": {"type": "integer"},
                  "unit": {"enum": ["days", "weeks", "months"]},
                  # an anchored window carries encoded (not flattened) value nodes
                  "from": {"type": "object"}, "to": {"type": "object"}}}
    atom = {"type": "object", "additionalProperties": False, "required": ["type"],
            "properties": {
                "type": {"enum": list(frame.ATOM_TYPES)},
                "field": {"enum": FIELDS},
                "cmp": {"enum": list(frame.CMPS)},
                "valueKind": {"enum": VALUE_KINDS},
                "value": {"type": "string"},
                "num": {"type": "number"},
                "what": {"enum": ["me", "null"]},
                "negated": {"type": "boolean"},
                "kind": {"enum": KINDS},
                "lits": {"type": "array", "items": {"type": "string"},
                         "minItems": 1, "maxItems": 4},
                "window": _ref("window"), "set": _ref("set"), "set2": _ref("set")}}
    pred = {"type": "object", "additionalProperties": False,
            "required": ["join", "atoms"],
            "properties": {"join": {"enum": ["none", "and", "or"]},
                           "atoms": {"type": "array", "items": _ref("atom"),
                                     "minItems": 1, "maxItems": 2}}}
    setd = {"type": "object", "additionalProperties": False, "required": ["source"],
            "properties": {
                "source": {"enum": ["kind", "ref"]},
                "kind": {"enum": KINDS}, "ref": {"enum": REFS},
                "ordinal": {"type": "integer"},
                "called": {"type": "string"}, "called2": {"type": "string"},
                "window": _ref("window"), "window2": _ref("window"),
                "filtersA": _ref("pred"), "filtersB": _ref("pred"),
                "filtersC": _ref("pred"),
                "walk1": {"enum": KINDS}, "walk2": {"enum": KINDS},
                "orderField": {"enum": FIELDS},
                "orderDir": {"enum": ["asc", "desc"]},
                "limit": {"type": "integer"},
                "combineOp": {"enum": ["and", "except"]},
                "combineRight": _ref("set")}}
    arg = {"type": "object", "additionalProperties": False,
           "required": ["name", "valueKind"],
           "properties": {"name": ARG_NAME,
                          "valueKind": {"enum": VALUE_KINDS},
                          "value": {"type": "string"},
                          "field": {"enum": FIELDS},
                          "set": _ref("set"), "set2": _ref("set")}}
    cmd = {"type": "object", "additionalProperties": False, "required": ["verb"],
           "properties": {"verb": {"enum": VERBS},
                          "args": {"type": "array", "items": _ref("arg"), "maxItems": 6},
                          "on": {"anyOf": [_ref("set"), {"type": "null"}]}}}
    return {"type": "object", "additionalProperties": False, "required": ["head"],
            "properties": {
                "head": {"enum": list(frame.FLAT_HEADS)},
                "aggField": {"enum": FIELDS},
                "refuseReason": {"enum": REASONS},
                "set": _ref("set"), "set2": _ref("set"),
                "cmds": {"type": "array", "items": _ref("cmd"),
                         "minItems": 1, "maxItems": 3}},
            "$defs": {"window": window, "atom": atom, "pred": pred, "set": setd,
                      "arg": arg, "cmd": cmd}}

# If the branch has since grown `experiments/qwen-sanity/mkschema.py` (a sibling lane's
# schema over the same frame), prefer it: one schema, one place to fix it.
SIBLING = os.path.join(REPO, "experiments", "qwen-sanity", "mkschema.py")
SCHEMA_SOURCE = "embedded in this notebook"
if os.path.exists(SIBLING):
    try:
        spec = importlib.util.spec_from_file_location("mkschema", SIBLING)
        mod = importlib.util.module_from_spec(spec)
        sys.path.insert(0, os.path.dirname(SIBLING))
        spec.loader.exec_module(mod)
        for fn in ("schema", "frame_json_schema", "build_schema", "make_schema"):
            if hasattr(mod, fn):
                frame_json_schema = getattr(mod, fn)
                SCHEMA_SOURCE = "%s:%s()" % (SIBLING, fn)
                break
    except Exception as exc:
        print("qwen-sanity/mkschema.py is present but did not import (%s); using the "
              "embedded schema." % exc)
print("schema from: %s" % SCHEMA_SOURCE)

SCHEMA = frame_json_schema()
with open(os.path.join(OUT, "frame.schema.json"), "w") as fh:
    json.dump(SCHEMA, fh, indent=2)

# ------------------------------------------------------------------ prove it
import jsonschema
VALIDATOR = jsonschema.Draft202012Validator(SCHEMA)
GOLD = frame.gold_turns()
bad = []
for row in GOLD:
    flat = frame.to_flat(frame.encode(check.parse(row["canonical"])))
    errs = list(VALIDATOR.iter_errors(flat))
    if errs:
        bad.append((row, errs[0].message))
print("gold wire frames accepted by the schema: %d/%d" % (len(GOLD) - len(bad), len(GOLD)))
for row, why in bad[:6]:
    print("   REJECT %s/%s t%s  %s\n          %s"
          % (row["corpus"], row["session"], row["turn"], why[:160], row["canonical"]))
if bad:
    raise Abort("The schema rejects %d gold wire frames. The SCHEMA is wrong, not the "
                "frame: every one of these is an object `to_flat` really produces, so a "
                "grammar built from this schema would make them undecodable and cap every "
                "score below the floor for a reason that is not the model." % len(bad))

sample = [r["flat"] for r in TRAIN[:4000]]
bad2 = sum(1 for f in sample if list(VALIDATOR.iter_errors(f)))
print("training frames accepted: %d/%d" % (len(sample) - bad2, len(sample)))
if bad2:
    raise Abort("The schema rejects %d of its own training targets." % bad2)


In [ ]:
import glob, os, shutil, subprocess

# llama.cpp is built ONCE and shared by every arm: it converts and quantises the merged
# adapters, and it is the thing that actually decodes the final runs -- i.e. the score is
# measured on the artefact that would ship, not on the fp16 training graph.
# Build from master: the Gated-DeltaNet / Mamba-2 operators the hybrid arms need are
# recent, and a stale build either refuses to load the architecture or loops.
with phase("llama.cpp clone"):
    if not os.path.isdir(LLAMA):
        sh(["git", "clone", "--depth", "1",
            "https://github.com/ggml-org/llama.cpp", LLAMA])
    print("llama.cpp", subprocess.run(["git", "-C", LLAMA, "rev-parse", "--short", "HEAD"],
                                      capture_output=True, text=True).stdout.strip())
    sh([sys.executable, "-m", "pip", "install", "-q", "-r",
        os.path.join(LLAMA, "requirements.txt")], check=False)

LLAMA_BUILD = os.path.join(LLAMA, "build")
def build_llama(cuda):
    args = ["cmake", "-B", LLAMA_BUILD, "-DCMAKE_BUILD_TYPE=Release",
            "-DLLAMA_CURL=OFF", "-DGGML_NATIVE=ON"]
    if cuda:
        # sm_60 is the P100, sm_75 the T4. Both, so the same build serves either.
        args += ["-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=60;75"]
    proc = sh(args, cwd=LLAMA, check=False, quiet=True)
    if proc.returncode != 0:
        print(proc.stdout[-3000:], proc.stderr[-3000:])
        return False
    proc = sh(["cmake", "--build", LLAMA_BUILD, "--config", "Release", "-j",
               str(os.cpu_count() or 4), "--target", "llama-quantize", "llama-server",
               "llama-cli"], cwd=LLAMA, check=False, quiet=True)
    if proc.returncode != 0:
        print(proc.stdout[-6000:], proc.stderr[-6000:])
        return False
    return True

with phase("llama.cpp build"):
    LLAMA_HAS_CUDA = False
    if LLAMA_CUDA:
        LLAMA_HAS_CUDA = build_llama(True)
        if not LLAMA_HAS_CUDA:
            print("\nThe CUDA build of llama.cpp FAILED. Falling back to a CPU build.\n"
                  "This is not fatal: a Q4_K_M model of ~1 B decodes at tens of tokens a\n"
                  "second on Kaggle's four cores, so the final decode takes longer but is\n"
                  "the same decode. The GGUF sizes and the scores are unaffected.")
            shutil.rmtree(LLAMA_BUILD, ignore_errors=True)
    if not LLAMA_HAS_CUDA and not build_llama(False):
        raise Abort("llama.cpp will not build at all. Send the output above back.")

def llama_bin(name):
    for pattern in (os.path.join(LLAMA_BUILD, "bin", name),
                    os.path.join(LLAMA_BUILD, name)):
        if os.path.exists(pattern):
            return pattern
    found = glob.glob(os.path.join(LLAMA_BUILD, "**", name), recursive=True)
    if not found:
        raise Abort("llama.cpp built but %r is not in %s" % (name, LLAMA_BUILD))
    return found[0]

LLAMA_SERVER = llama_bin("llama-server")
LLAMA_QUANTIZE = llama_bin("llama-quantize")
CONVERT = os.path.join(LLAMA, "convert_hf_to_gguf.py")
if not os.path.exists(CONVERT):
    hits = glob.glob(os.path.join(LLAMA, "**", "convert*hf*gguf*.py"), recursive=True)
    if not hits:
        raise Abort("llama.cpp has no convert_hf_to_gguf.py; the export step cannot run.")
    CONVERT = hits[0]
print("server   ", LLAMA_SERVER, "(cuda: %s)" % LLAMA_HAS_CUDA)
print("quantize ", LLAMA_QUANTIZE)
print("convert  ", CONVERT)

# ------------------------------------------------------- JSON schema -> GBNF
GBNF_PATH = os.path.join(OUT, "frame.gbnf")
GBNF = None
conv = [p for p in glob.glob(os.path.join(LLAMA, "**", "json_schema_to_grammar.py"),
                             recursive=True)]
if GRAMMAR_MODE == "gbnf" and conv:
    try:
        spec = importlib.util.spec_from_file_location("j2g", conv[0])
        j2g = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(j2g)
        # upstream has moved this helper around; try the spellings in turn.
        for fn in ("json_schema_to_gbnf", "convert_json_schema_to_gbnf"):
            if hasattr(j2g, fn):
                GBNF = getattr(j2g, fn)(SCHEMA)
                break
        if GBNF is None:
            try:
                conv_obj = j2g.SchemaConverter(prop_order={}, allow_fetch=False,
                                              dotall=False, raw_pattern=False)
            except TypeError:
                conv_obj = j2g.SchemaConverter({}, False, False, False)
            conv_obj.visit(SCHEMA, "")
            GBNF = conv_obj.format_grammar()
    except Exception as exc:
        print("llama.cpp's json_schema_to_grammar did not convert the schema (%s: %s).\n"
              "Falling back to GRAMMAR_MODE='json_schema': llama-server does the same\n"
              "conversion in C++ and the constraint is identical." % (type(exc).__name__, exc))
        GBNF = None
if GBNF:
    with open(GBNF_PATH, "w") as fh:
        fh.write(GBNF)
    print("GBNF written: %s (%.1f KiB, %d rules)"
          % (GBNF_PATH, len(GBNF) / 1024.0, GBNF.count("::=")))
    if "root" not in GBNF:
        raise Abort("the generated GBNF has no root rule")
    GRAMMAR_MODE_EFFECTIVE = "gbnf"
else:
    GRAMMAR_MODE_EFFECTIVE = "json_schema"
    print("constraint will be sent to llama-server as `json_schema`.")


## The sweep

Everything above this point is **shared**: one clone, one Rust build, one wiring proof, one
verified training set, one leakage gate, one schema, one llama.cpp. Everything below is
**per arm**, and one arm's failure never costs another's result.

Each arm does, in order:

1. load the base in fp16 with SDPA attention;
2. **discover** its real `nn.Linear` leaves and resolve LoRA targets against them — the
   per-family list in CONFIG is only a preference. A wrong target list trains almost
   nothing and says nothing about it, so the **trainable fraction is asserted into
   [0.05%, 5%]** and the arm aborts otherwise;
3. train, checkpointing and evaluating **every epoch** into `/kaggle/working`;
4. merge the chosen adapter and convert to `Q4_K_M` GGUF — and if llama.cpp cannot convert
   that architecture, **the arm still finishes**: it decodes through transformers under the
   same JSON-schema constraint and the GGUF column says `unsupported`;
5. decode all 434 gold turns under the grammar, **saving raw output before anything parses
   it**, then render with the repository's own `render_frames.py` and score with
   `run-model`.

A malformed, empty or unrenderable output is a failed turn and is counted as one. Nothing
is repaired.


In [ ]:
import json, numpy as np, random, torch

# ------------------------------------------------------------------ the instruction
# Doctrine.swift is GENERATED from the grammar's lexicon by gen_vocab.instructions_text();
# reading the generator rather than the .swift file means the prompt cannot drift from the
# ontology the scorer uses.
DOCTRINE_FULL = gen_vocab.instructions_text()
DOCTRINE_SHORT = "\n".join([
    "Fill in a form describing what one sentence to a private vault MEANS. Never write SQL.",
    "head: show | count | sum | min | max | project | balance | same | write | nothing | refuse",
    "set.source: kind (name a board) or ref (it, them, that one, the other one, the earlier"
    " one, the last thing I added, ordinal).",
    "Steps: called, that (filters), during (window), walk, ordered_by, limit.",
    "refuse when the vault cannot hold the request; nothing when the person withdrew.",
    "COPY, NEVER INVENT: every quoted value must appear word for word in the sentence or in"
    " the previous canonical.",
])
SYSTEM_TEXT = {"full": DOCTRINE_FULL, "short": DOCTRINE_SHORT, "none": None}[
    SYSTEM_PROMPT_MODE]

_n = len(DOCTRINE_FULL.split())
print("Doctrine.swift is %d words (~%d tokens); SYSTEM_PROMPT_MODE = %r.\n"
      "Arithmetic, because this departs from the brief: a ~700-token fixed prefix over\n"
      "45 722 rows is ~32 M tokens an epoch against ~5.5 M for the rest of the sequence.\n"
      "At ~8 x N FLOP per training token with gradient checkpointing and ~5 TFLOP/s\n"
      "realised on a P100, the doctrine ALONE is ~11 h an epoch for a 0.8 B model -- the\n"
      "run cannot finish inside a 12 h session, let alone four arms. A fine-tuned model\n"
      "does not need the instruction block: it has 45 722 worked examples of the same\n"
      "thing. Set SYSTEM_PROMPT_MODE='full' and MAX_SEQ_LEN=1536 if you want it and have\n"
      "the quota; the prompt is built in ONE place so train, eval and decode move together."
      % (_n, int(_n * 1.35), SYSTEM_PROMPT_MODE))

IM_START, IM_END = "<|im_start|>", "<|im_end|>"

def user_text(prev_canonical, request):
    prev = (prev_canonical or "NONE").strip() or "NONE"
    return "previous: %s\nsaid: %s" % (prev, request.strip())

def make_prompt_fn(tok):
    """One prompt builder per base. The SAME string must come out at training time
    (transformers) and at decode time (llama.cpp gets a raw string), so the checkpoint's
    own chat template is authoritative and a hand-built ChatML is only the fallback."""
    def chatml(prev, request):
        parts = []
        if SYSTEM_TEXT:
            parts.append("%ssystem\n%s%s\n" % (IM_START, SYSTEM_TEXT, IM_END))
        parts.append("%suser\n%s%s\n" % (IM_START, user_text(prev, request), IM_END))
        parts.append("%sassistant\n" % IM_START)
        return "".join(parts)

    def templated(prev, request):
        msgs = []
        if SYSTEM_TEXT:
            msgs.append({"role": "system", "content": SYSTEM_TEXT})
        msgs.append({"role": "user", "content": user_text(prev, request)})
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    try:
        probe = templated("NONE", "show me tomorrow")
        if not isinstance(probe, str) or not probe:
            raise ValueError("empty")
        return templated, "checkpoint chat template"
    except Exception as exc:
        print("   apply_chat_template unusable (%s); using hand-built ChatML." % exc)
        return chatml, "hand-built ChatML"

def target_text(flat):
    """The frame as the model must emit it: compact, key-sorted, one line."""
    return json.dumps(flat, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

# ------------------------------------------------------------------ tokenising
def encode_rows(rows, tok, build_prompt, stop_id, label):
    prompts = [build_prompt(r["prev"], r["request"]) for r in rows]
    answers = [target_text(r["flat"]) for r in rows]
    p_ids = tok(prompts, add_special_tokens=False)["input_ids"]
    a_ids = tok(answers, add_special_tokens=False)["input_ids"]
    kept, too_long, lengths = [], 0, []
    for pid, aid in zip(p_ids, a_ids):
        ids = pid + aid + [stop_id]
        lengths.append(len(ids))
        if len(ids) > MAX_SEQ_LEN:
            too_long += 1
            continue
        kept.append({"input_ids": ids,
                     "labels": [-100] * len(pid) + aid + [stop_id]})
    lengths = np.array(lengths)
    print("   %-12s %6d rows | tokens p50 %d p90 %d p99 %d max %d | over %d: %d (%.2f%%)"
          % (label, len(rows), np.percentile(lengths, 50), np.percentile(lengths, 90),
             np.percentile(lengths, 99), lengths.max(), MAX_SEQ_LEN, too_long,
             100.0 * too_long / max(1, len(rows))))
    if too_long > 0.02 * len(rows):
        print("   WARNING: >2%% of rows do not fit MAX_SEQ_LEN=%d. Raise it rather than "
              "training on a biased-short subset." % MAX_SEQ_LEN)
    return kept

class FrameDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return self.rows[i]

def make_collate(pad_id):
    def collate(batch):
        width = max(len(b["input_ids"]) for b in batch)
        input_ids = torch.full((len(batch), width), pad_id, dtype=torch.long)
        labels = torch.full((len(batch), width), -100, dtype=torch.long)
        attn = torch.zeros((len(batch), width), dtype=torch.long)
        for i, b in enumerate(batch):
            n = len(b["input_ids"])
            input_ids[i, :n] = torch.tensor(b["input_ids"], dtype=torch.long)
            labels[i, :n] = torch.tensor(b["labels"], dtype=torch.long)
            attn[i, :n] = 1
        return {"input_ids": input_ids, "labels": labels, "attention_mask": attn}
    return collate

# ------------------------------------------------------------------ validation
ABSTAIN_HEADS = {"refuse", "nothing"}
_rng = random.Random(SEED)

def subset(rows, k):
    return rows if not k or k >= len(rows) else _rng.sample(rows, k)

EVAL_SMALL = [("distill_val", subset(VAL_DISTILL, EVAL_N)),
              ("val_template", subset(VAL_TEMPLATE, EVAL_N))]
EVAL_FULL = [("distill_val", VAL_DISTILL), ("val_template", VAL_TEMPLATE)]
print("\nper-epoch eval: %s" % ", ".join("%s n=%d" % (k, len(v)) for k, v in EVAL_SMALL))
print("final eval:     %s" % ", ".join("%s n=%d" % (k, len(v)) for k, v in EVAL_FULL))
print("The two files are NEVER pooled: distill_val is held out by SKELETON (unseen shapes,\n"
      "real English); val.jsonl is held-out TEMPLATES from the old generator. A model that\n"
      "scores well on the second and badly on the first has learnt the generator again.")

@torch.no_grad()
def generate_frames(model, tok, build_prompt, rows, stop_ids, batch_size=16):
    """Greedy, unconstrained. Raw strings come back exactly as decoded."""
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.eval()
    side = tok.padding_side
    tok.padding_side = "left"
    out = []
    try:
        for start in range(0, len(rows), batch_size):
            chunk = rows[start:start + batch_size]
            enc = tok([build_prompt(r["prev"], r["request"]) for r in chunk],
                      return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
            gen = model.generate(**enc, max_new_tokens=N_PREDICT, do_sample=False,
                                 num_beams=1, pad_token_id=tok.pad_token_id,
                                 eos_token_id=stop_ids)
            out.extend(tok.batch_decode(gen[:, enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True))
    finally:
        tok.padding_side = side
        model.config.use_cache = False
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False})
        model.train()
    return out

def score_frames(rows, raws):
    """Tree-equality against gold, with abstentions kept on their own row."""
    n = len(rows)
    parsed = exact = 0
    gold_abs = abs_right = non_abs = non_abs_right = false_abs = 0
    for row, raw in zip(rows, raws):
        is_abs = row["head"] in ABSTAIN_HEADS
        haystack = "%s %s" % (row["prev"], row["request"])
        canonical, _reason = render_frames.render_one(raw.strip(), haystack)
        ok = False
        if canonical:
            parsed += 1
            try:
                ok = check.parse(canonical) == check.parse(row["target"])
            except Exception:
                ok = False
        try:
            pred_head = json.loads(raw.strip()).get("head")
        except Exception:
            pred_head = None
        if is_abs:
            gold_abs += 1
            abs_right += int(ok)
        else:
            non_abs += 1
            non_abs_right += int(ok)
            false_abs += int(pred_head in ABSTAIN_HEADS)
        exact += int(ok)
    return {"n": n,
            "parse_rate": 100.0 * parsed / max(1, n),
            "exact": 100.0 * exact / max(1, n),
            "n_answer": non_abs,
            "answer_exact": 100.0 * non_abs_right / max(1, non_abs),
            "n_abstain": gold_abs,
            "abstain_exact": (100.0 * abs_right / gold_abs) if gold_abs else None,
            "false_abstain": 100.0 * false_abs / max(1, non_abs)}

def evaluate(model, tok, build_prompt, rows, stop_ids, label, batch_size=16):
    raws = generate_frames(model, tok, build_prompt, rows, stop_ids, batch_size)
    m = score_frames(rows, raws)
    m["label"] = label
    print("   %-26s n=%-5d parse %5.1f%%  exact %5.1f%%  | answers %d: %5.1f%%  "
          "abstentions %s: %s  false-abstain %4.1f%%"
          % (label, m["n"], m["parse_rate"], m["exact"], m["n_answer"],
             m["answer_exact"], m["n_abstain"],
             ("%5.1f%%" % m["abstain_exact"]) if m["abstain_exact"] is not None else "  n/a",
             m["false_abstain"]))
    return m


In [ ]:
import collections, gc, math, os, re, shutil, torch, transformers
from transformers import (AutoConfig, AutoTokenizer, Trainer, TrainingArguments,
                          TrainerCallback)
from peft import LoraConfig, get_peft_model

def save_result(base, results):
    """One JSON per arm, written after every stage, so an eviction loses at most the
    stage in flight -- and so the comparison table can assemble across sessions."""
    results["slug"] = base["slug"]
    results["hf_id"] = base["hf_id"]
    results["licence"] = base["licence"]
    results["family"] = base["family"]
    results["approx_params"] = base["approx_params"]
    results["train_params"] = base["train_params"]
    results["config"] = {"epochs": EPOCHS, "lr": LR, "lora_r": LORA_R,
                         "max_seq_len": MAX_SEQ_LEN,
                         "system_prompt_mode": SYSTEM_PROMPT_MODE,
                         "train_rows": len(TRAIN), "smoke": SMOKE, "seed": SEED}
    with open(os.path.join(RESULTS, base["slug"] + ".json"), "w") as fh:
        json.dump(results, fh, indent=2, default=str)

def load_base(base):
    """Load the checkpoint in fp16 with SDPA. The class comes from config.architectures;
    the auto classes are the fallback, in order."""
    mid = base["hf_id"]
    cfg = AutoConfig.from_pretrained(mid)
    errors = []
    for name in list(getattr(cfg, "architectures", None) or []):
        cls = getattr(transformers, name, None)
        if cls is None:
            errors.append("%s: not in this transformers" % name)
            continue
        try:
            return cls.from_pretrained(mid, dtype=DTYPE,
                                       attn_implementation=ATTN_IMPL), name, cfg
        except Exception as exc:
            errors.append("%s: %s" % (name, str(exc)[:200]))
    for auto in ("AutoModelForImageTextToText", "AutoModelForCausalLM",
                 "AutoModelForVision2Seq", "AutoModel"):
        cls = getattr(transformers, auto, None)
        if cls is None:
            continue
        try:
            return cls.from_pretrained(mid, dtype=DTYPE,
                                       attn_implementation=ATTN_IMPL), auto, cfg
        except Exception as exc:
            errors.append("%s: %s" % (auto, str(exc)[:200]))
    raise Abort("cannot load %s:\n  %s" % (mid, "\n  ".join(errors)))

def resolve_targets(model, base):
    """Discover the real nn.Linear leaves of the TEXT stack, then resolve LoRA targets
    against them. The per-family list in CONFIG is a preference, never an assumption."""
    prefix = ""
    for name, _m in model.named_modules():
        if re.search(r"(^|\.)(language_model|text_model|model\.text_model)$", name):
            prefix = name
            break
    leaves = collections.Counter()
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        if prefix and not name.startswith(prefix + "."):
            continue
        leaf = name.rsplit(".", 1)[-1]
        if leaf in ("lm_head", "score", "embed_out"):
            continue
        leaves[leaf] += 1
    print("   text stack at %r; nn.Linear leaves:" % (prefix or "<whole model>"))
    for leaf, count in leaves.most_common():
        print("      %-20s %4d %s"
              % (leaf, count, "<- preferred" if leaf in base["lora_target_modules"] else ""))
    preferred = [s for s in base["lora_target_modules"] if leaves.get(s)]
    chosen, how = preferred, "the family list in CONFIG"
    if not preferred:
        chosen, how = list(leaves), "EVERY linear leaf (the family list matched nothing)"
        print("   NOTE: none of %s exists here. Falling back to every linear leaf."
              % base["lora_target_modules"])
    missed = [s for s in leaves if s not in chosen]
    if missed:
        print("   not targeted: %s" % ", ".join(sorted(missed)))
    if not chosen:
        raise Abort("no nn.Linear modules found under the text stack of %s" % base["hf_id"])
    regex = (("^%s\\." % re.escape(prefix)) if prefix else "") + \
            r".*\.(" + "|".join(re.escape(s) for s in chosen) + r")$"
    return regex, chosen, how, sum(leaves[s] for s in chosen)

def attach_lora(model, base):
    regex, chosen, how, count = resolve_targets(model, base)
    print("   LoRA targets (%s): %s  -> %d modules" % (how, ", ".join(chosen), count))
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
        task_type="CAUSAL_LM", target_modules=regex))
    # fp16 master weights diverge under a GradScaler: keep the trainable parameters in
    # fp32 and let autocast handle the forward. There is no bf16 on sm_60/sm_75 to paper
    # over it.
    for _n, p in model.named_parameters():
        if p.requires_grad:
            p.data = p.data.float()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100.0 * trainable / total
    print("   trainable %s / %s = %.3f%%" % (f"{trainable:,}", f"{total:,}", pct))
    if not (MIN_TRAINABLE_PCT <= pct <= MAX_TRAINABLE_PCT):
        raise Abort(
            "trainable fraction %.4f%% is outside [%.2f%%, %.2f%%] for %s.\n"
            "Below the floor means the target modules are wrong and this arm would train\n"
            "almost nothing while reporting a number -- the exact silent failure the\n"
            "bake-off exists to avoid. Above the ceiling means this is not LoRA. The real\n"
            "leaf names are printed above; put the right ones in this base's\n"
            "lora_target_modules." % (pct, MIN_TRAINABLE_PCT, MAX_TRAINABLE_PCT,
                                      base["hf_id"]))
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model.config.use_cache = False
    return model, {"targets": chosen, "how": how, "modules": count,
                   "trainable": trainable, "total": total, "trainable_pct": pct}

class NanGuard(TrainerCallback):
    """A non-finite loss on a GPU with no bf16 is not a blip to ride out: every number
    afterwards is noise. Stop the ARM, keep the others."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        loss = (logs or {}).get("loss")
        if loss is not None and not math.isfinite(loss):
            raise Abort("loss became %r in fp16. This base overflows at fp16 and this GPU "
                        "has no bf16. Lower LR, or run this arm on an sm_80+ GPU. The "
                        "number it would produce is meaningless." % loss)

def train_arm(base, model, tok, build_prompt, stop_ids, train_ds, collate, results):
    epochs_log = []

    def run_eval(tag, sets):
        row = {"epoch": tag}
        for name, rows in sets:
            row[name] = evaluate(model, tok, build_prompt, rows, stop_ids,
                                 "%s @ %s" % (name, tag))
        epochs_log.append(row)
        results["epochs"] = epochs_log
        save_result(base, results)
        return row

    class PerEpoch(TrainerCallback):
        """Checkpoint AND evaluate every epoch: the deliverable is the curve, and a 12 h
        session eviction must not cost the run."""
        def on_epoch_end(self, args, state, control, **kwargs):
            epoch = int(round(state.epoch))
            target = os.path.join(CKPT, base["slug"], "epoch-%d" % epoch)
            os.makedirs(target, exist_ok=True)
            model.save_pretrained(target)
            print("\n   [%s epoch %d] adapter -> %s" % (base["slug"], epoch, target))
            with phase("%s eval epoch %d" % (base["slug"], epoch)):
                run_eval(epoch, EVAL_SMALL)
            return control

    args = TrainingArguments(
        output_dir=os.path.join(SCRATCH, "trainer", base["slug"]),
        overwrite_output_dir=True, num_train_epochs=EPOCHS,
        per_device_train_batch_size=base["per_device_bs"],
        gradient_accumulation_steps=base["grad_accum"], learning_rate=LR,
        warmup_ratio=WARMUP_RATIO, lr_scheduler_type="cosine", weight_decay=0.0,
        max_grad_norm=1.0, logging_steps=50, save_strategy="no", eval_strategy="no",
        fp16=not BF16_OK, bf16=BF16_OK, optim="adamw_torch",
        dataloader_num_workers=2, remove_unused_columns=False, report_to=[], seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      data_collator=collate, callbacks=[PerEpoch(), NanGuard()])

    print("\n   epoch 0 -- the untrained baseline. A fine-tune that does not move this is")
    print("   not learning the task; a high baseline would be leakage, not skill.")
    with phase("%s eval epoch 0" % base["slug"]):
        run_eval(0, EVAL_SMALL)
    with phase("%s train" % base["slug"]):
        trainer.train()
    with phase("%s eval final (full)" % base["slug"]):
        run_eval("final", EVAL_FULL)
    return epochs_log

def pick_epoch(epochs_log):
    """EXPORT_EPOCH -> the adapter directory to merge."""
    numbered = [e for e in epochs_log if isinstance(e["epoch"], int) and e["epoch"] > 0]
    if not numbered:
        raise Abort("no epoch checkpoints were written")
    if EXPORT_EPOCH == "last":
        return numbered[-1]["epoch"]
    if isinstance(EXPORT_EPOCH, int):
        return EXPORT_EPOCH
    return max(numbered, key=lambda e: e["distill_val"]["exact"])["epoch"]

def free_gpu(*objs):
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
import json, os, shutil, signal, socket, subprocess, time
import concurrent.futures as futures
import requests

# ============================================================== merge + GGUF export
def merge_and_export(base, epoch, tok):
    """Merge the chosen adapter and convert to Q4_K_M. A conversion failure is a RESULT
    (gguf: unsupported), never the end of the arm."""
    from peft import PeftModel
    out = {"epoch": epoch, "gguf": None, "gguf_bytes": None, "f16_bytes": None,
           "supported": False, "why": None}
    merged_dir = os.path.join(SCRATCH, "merged", base["slug"])
    shutil.rmtree(merged_dir, ignore_errors=True)
    os.makedirs(merged_dir, exist_ok=True)
    adapter = os.path.join(CKPT, base["slug"], "epoch-%d" % epoch)
    with phase("%s merge adapter (epoch %d)" % (base["slug"], epoch)):
        plain, _cls, _cfg = load_base(base)
        merged = PeftModel.from_pretrained(plain, adapter).merge_and_unload()
        merged.save_pretrained(merged_dir, safe_serialization=True)
        tok.save_pretrained(merged_dir)
        # the converter wants the tokenizer/processor files beside the weights
        try:
            from huggingface_hub import snapshot_download
            src = snapshot_download(base["hf_id"], allow_patterns=[
                "tokenizer*", "vocab*", "merges*", "*.jinja", "preprocessor_config.json",
                "special_tokens_map.json"])
            for name in os.listdir(src):
                dst = os.path.join(merged_dir, name)
                if not os.path.exists(dst):
                    shutil.copy(os.path.join(src, name), dst)
        except Exception as exc:
            print("   (could not copy auxiliary tokenizer files: %s)" % exc)
        free_gpu(plain, merged)

    f16 = os.path.join(SCRATCH, "%s-f16.gguf" % base["slug"])
    quant = os.path.join(OUT, "%s-%s.gguf" % (base["slug"], GGUF_QUANT.lower()))
    with phase("%s gguf convert + quantise" % base["slug"]):
        proc = sh([sys.executable, CONVERT, merged_dir, "--outfile", f16,
                   "--outtype", "f16"], check=False, quiet=True)
        if proc.returncode != 0 or not os.path.exists(f16):
            tail = (proc.stdout[-2500:] + proc.stderr[-2500:]).strip()
            print("   GGUF conversion FAILED for %s:\n%s" % (base["hf_id"], tail[-2500:]))
            out["why"] = "convert_hf_to_gguf.py rejected this architecture"
            out["convert_log"] = tail[-2000:]
            return out, merged_dir
        proc = sh([LLAMA_QUANTIZE, f16, quant, GGUF_QUANT], check=False, quiet=True)
        if proc.returncode != 0 or not os.path.exists(quant):
            print((proc.stdout + proc.stderr)[-2500:])
            out["why"] = "llama-quantize failed"
            return out, merged_dir
    out.update(supported=True, gguf=quant, gguf_bytes=os.path.getsize(quant),
               f16_bytes=os.path.getsize(f16))
    print("   %-10s f16 %.0f MB  ->  %s %.0f MB   (the mobile budget is the point)"
          % (base["slug"], out["f16_bytes"] / 1e6, GGUF_QUANT, out["gguf_bytes"] / 1e6))
    os.remove(f16)
    return out, merged_dir

# ============================================================== llama-server decode
def free_port():
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port

class LlamaServer:
    def __init__(self, gguf, slots):
        self.port = free_port()
        cmd = [LLAMA_SERVER, "-m", gguf, "--host", "127.0.0.1", "--port", str(self.port),
               "-c", str(max(2048, MAX_SEQ_LEN * 4)), "-np", str(slots),
               "--no-webui", "-t", str(os.cpu_count() or 4)]
        if LLAMA_HAS_CUDA:
            cmd += ["-ngl", "99"]
        self.proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                     stderr=subprocess.STDOUT, text=True)
        self.url = "http://127.0.0.1:%d" % self.port
    def __enter__(self):
        for _ in range(600):
            if self.proc.poll() is not None:
                raise Abort("llama-server exited before it was ready:\n"
                            + (self.proc.stdout.read() or "")[-3000:])
            try:
                if requests.get(self.url + "/health", timeout=2).status_code == 200:
                    return self
            except Exception:
                pass
            time.sleep(1)
        raise Abort("llama-server did not come up in 10 minutes")
    def __exit__(self, *exc):
        self.proc.send_signal(signal.SIGINT)
        try:
            self.proc.wait(timeout=30)
        except Exception:
            self.proc.kill()
    def complete(self, prompt):
        body = {"prompt": prompt, "temperature": 0.0, "top_k": 1,
                "n_predict": N_PREDICT, "cache_prompt": True,
                "stop": ["<|im_end|>", "<|endoftext|>", "<end_of_turn>", "<eos>"]}
        if GRAMMAR_MODE_EFFECTIVE == "gbnf":
            body["grammar"] = GBNF
        else:
            body["json_schema"] = SCHEMA
        t0 = time.time()
        r = requests.post(self.url + "/completion", json=body, timeout=600)
        r.raise_for_status()
        return r.json().get("content", ""), (time.time() - t0) * 1000.0

# ---------------------------------------------- the constrained transformers fallback
def transformers_decoder(model, tok, stop_ids):
    """Used when llama.cpp cannot convert the architecture. Constrained with llguidance
    against the SAME schema when it is available; when it is not, the decode is
    UNCONSTRAINED and the result says so -- it never pretends."""
    processors = None
    constrained = False
    if HAVE_LLGUIDANCE:
        try:
            import llguidance
            from llguidance.hf import from_tokenizer
            from llguidance.torch import allocate_token_bitmask, fill_next_token_bitmask
            from transformers import LogitsProcessor, LogitsProcessorList
            ll_tok = from_tokenizer(tok)
            grammar = llguidance.LLMatcher.grammar_from_json_schema(SCHEMA)

            class LLG(LogitsProcessor):
                def __init__(self):
                    self.matcher = None
                    self.mask = allocate_token_bitmask(1, ll_tok.vocab_size)
                    self.start = None
                def __call__(self, input_ids, scores):
                    if self.matcher is None or self.start is None:
                        self.matcher = llguidance.LLMatcher(ll_tok, grammar)
                        self.start = input_ids.shape[1]
                    else:
                        self.matcher.consume_token(int(input_ids[0, -1]))
                    fill_next_token_bitmask(self.matcher, self.mask, 0)
                    allowed = torch.zeros_like(scores, dtype=torch.bool)
                    bits = self.mask[0]
                    for i in range(scores.shape[1]):
                        if bits[i >> 5].item() >> (i & 31) & 1:
                            allowed[0, i] = True
                    return scores.masked_fill(~allowed, float("-inf"))
            processors = LogitsProcessorList([LLG()])
            constrained = True
        except Exception as exc:
            print("   llguidance constraint unavailable (%s: %s); this arm decodes "
                  "UNCONSTRAINED and is labelled so." % (type(exc).__name__, exc))

    @torch.no_grad()
    def decode(prompt):
        model.config.use_cache = True
        model.eval()
        enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        t0 = time.time()
        kwargs = {}
        if processors is not None:
            kwargs["logits_processor"] = processors
            processors[0].matcher = None
            processors[0].start = None
        gen = model.generate(**enc, max_new_tokens=N_PREDICT, do_sample=False,
                             num_beams=1, pad_token_id=tok.pad_token_id,
                             eos_token_id=stop_ids, **kwargs)
        text = tok.decode(gen[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        return text, (time.time() - t0) * 1000.0
    return decode, constrained

# ============================================================== the 434 gold turns
def gold_sessions():
    rows = frame.gold_turns()
    by = {}
    for row in rows:
        by.setdefault((row["corpus"], row["session"]), []).append(row)
    for turns in by.values():
        turns.sort(key=lambda r: r["turn"])
    return by

def decode_corpus(base, build_prompt, complete, mode, raw_path, parallel):
    """free: the model's OWN previous canonical is fed back. teacher: the gold one.
    Raw model output is written BEFORE anything parses it."""
    sessions = gold_sessions()
    if SMOKE:
        sessions = dict(list(sessions.items())[:40])
    records, lock = [], __import__("threading").Lock()

    def run_session(key, turns):
        out, previous = [], "NONE"
        for row in turns:
            prompt = build_prompt(previous if mode == "free" else row.get("prev_gold",
                                                                          previous),
                                  row["request"])
            try:
                raw, ms = complete(prompt)
                err = ""
            except Exception as exc:
                raw, ms, err = "", 0.0, "%s: %s" % (type(exc).__name__, str(exc)[:200])
            rec = {"corpus": row["corpus"], "session": row["session"],
                   "turn": row["turn"], "mode": mode, "model": base["slug"],
                   "request": row["request"], "prev_used": previous,
                   "stageA": "", "stageB": raw, "error": err,
                   "ms_a": 0, "ms_b": ms, "canonical": ""}
            out.append(rec)
            if mode == "free":
                canonical, _why = render_frames.render_one(raw.strip(),
                                                           "%s %s" % (previous,
                                                                      row["request"]))
                previous = canonical or "NONE"
            else:
                previous = row["canonical"]
        with lock:
            records.extend(out)
        return len(out)

    # sessions are independent; turns inside one are not. Parallel across sessions.
    keys = sorted(sessions)
    for row_key in keys:
        turns = sessions[row_key]
        prev = "NONE"
        for row in turns:                       # teacher mode's gold previous canonical
            row["prev_gold"] = prev
            prev = row["canonical"]
    with futures.ThreadPoolExecutor(max_workers=parallel) as pool:
        done = 0
        for n in pool.map(lambda k: run_session(k, sessions[k]), keys):
            done += n
            if done % 50 < n:
                print("      %d/%d turns" % (done, sum(len(v) for v in sessions.values())),
                      flush=True)
    records.sort(key=lambda r: (r["corpus"], r["session"], r["turn"]))
    with open(raw_path, "w", encoding="utf-8") as fh:
        for rec in records:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
    ms = [r["ms_b"] for r in records if r["ms_b"]]
    print("   raw -> %s (%d turns, %.0f ms/turn median)"
          % (raw_path, len(records), sorted(ms)[len(ms) // 2] if ms else 0))
    return records

def render_and_score(base, mode, raw_path):
    """The repository's own renderer, then the repository's own scorer. Nothing here
    repairs a model string: an empty or unrenderable frame is a failed turn."""
    rendered = os.path.join(OUT, "%s-%s.jsonl" % (base["slug"], mode))
    sh([sys.executable, os.path.join(AFM, "render_frames.py"),
        "--in", raw_path, "--out", rendered], cwd=AFM)
    reasons_path = rendered + ".reasons.jsonl"
    total = ok = 0
    reasons = collections.Counter()
    with open(reasons_path, encoding="utf-8") as fh:
        for line in fh:
            row = json.loads(line)
            total += 1
            if row["canonical"]:
                ok += 1
            else:
                reasons[(row["reason"] or "?").split(":")[0]] += 1
    scores = score_with_run_model(rendered) if not SMOKE else {}
    return {"rendered": rendered, "raw": raw_path,
            "parse_rate": 100.0 * ok / max(1, total), "turns": total,
            "failure_reasons": dict(reasons), "scores": scores}


## END OF THE DRAFT — the notebook stops here

Three cells were planned and **never written**, because fine-tuning was parked:

1. **The sweep driver** — a loop over `RUNNABLE` calling, per arm and inside one
   `try/except` so that one arm's failure never costs another's result:
   `load_base` -> `attach_lora` -> `make_prompt_fn` -> `encode_rows` -> `train_arm` ->
   `pick_epoch` -> `merge_and_export` -> (`LlamaServer` or `transformers_decoder`) ->
   `decode_corpus` for each of `DECODE_MODES` -> `render_and_score` -> `save_result`.
   Every one of those functions is defined above; only the loop is missing.
2. **The comparison table** — read `out/results/*.json`, one row per base, ordered by
   `PRIORITY`: strict sessions suite/blind/holdout against the floor (53/120, 34/60,
   31/78), graded F1, parse rate, `Q4_K_M` size in MB, train GPU-minutes, decode ms/turn,
   licence. Plus `summary.md` next to it.
3. **The optional results push** (`PUSH_RESULTS`, off by default, needs a token with write
   scope).

Nothing above has been executed. See the PARKED header.
